# 1. Load Metadata and Select References


In [ ]:
from pathlib import Path
import pandas as pd
import re

Root = Path.cwd()
if not (Root / "Formulation").exists():
    Root = Root.parent

CsvPath = Root / "Formulation" / "Literature" / "SImilar40" / "selected_scores_20260209_164223.csv"
BibPath = Root / "Formulation" / "Literature" / "SImilar40" / "knapsack_20260209_165848.bib"

DataFrame = pd.read_csv(CsvPath, usecols=["title", "abstract"], encoding="utf-8")
BibText = BibPath.read_text(encoding="utf-8")

KeyMatches = re.findall(r"@\w+\{([^,]+),", BibText)
AvailableKeys = set(KeyMatches)

PreferredKeys = [
    "büyüktahtakın_scenario_2023",
    "bos_distributionally_2024",
    "ding_balancing_2022",
    "chen_sample_2022",
    "larsen_fast_2024",
    "rezaeian_assignment_2024",
]

SelectedKeys = [Key for Key in PreferredKeys if Key in AvailableKeys]
SelectedTitles = DataFrame[DataFrame["title"].str.contains("knapsack|assignment|stochastic", case=False, na=False)]

SelectedKeys, SelectedTitles.head(5)


# 2. Update `Formulation.tex` from `Formulation_SJ.tex`


In [ ]:
from pathlib import Path

Root = Path.cwd()
if not (Root / "Formulation").exists():
    Root = Root.parent

FormulationPath = Root / "Formulation" / "Formulation.tex"
SourcePath = Root / "Collab" / "Formulation_SJ.tex"

FormulationText = FormulationPath.read_text(encoding="utf-8")
SourceText = SourcePath.read_text(encoding="utf-8")

if "Stochastic Multiple Knapsack Extension" in FormulationText:
    "Formulation.tex already includes the updated knapsack formulation."
else:
    MarkerStart = SourceText.find("\\section*{Problem Description}")
    MarkerEnd = SourceText.find("\\end{document}")
    Snippet = SourceText[MarkerStart:MarkerEnd].strip()
    "Formulation update required. Snippet extracted:", Snippet[:500]


# 3. Update `SLM.tex` with New Content and Citations


In [ ]:
from pathlib import Path

Root = Path.cwd()
if not (Root / "Presentation").exists():
    Root = Root.parent

SlidesPath = Root / "Presentation" / "SLM.tex"
SlidesText = SlidesPath.read_text(encoding="utf-8")

if "mkp_structure.png" in SlidesText and "Stochastic Multiple Knapsack" in SlidesText:
    "SLM.tex already includes updated slides and citations."
else:
    "SLM.tex update required."


# 4. Generate Stochastic Knapsack Figures


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

Root = Path.cwd()
if not (Root / "Presentation").exists():
    Root = Root.parent

OutputDir = Root / "Presentation"
OutputDir.mkdir(parents=True, exist_ok=True)

RandomState = np.random.default_rng(42)
AgentCount = 3
TaskCount = 8

AgentLabels = [f"Agent {Index + 1}" for Index in range(AgentCount)]
TaskLabels = [f"Task {Index + 1}" for Index in range(TaskCount)]
Capacities = RandomState.uniform(5.5, 7.5, size=AgentCount)
Sizes = RandomState.uniform(0.6, 2.0, size=TaskCount)
Returns = RandomState.uniform(2.0, 8.0, size=TaskCount)
Assignments = RandomState.integers(0, AgentCount, size=TaskCount)

AgentX = 0.82
TaskX = 0.18
AgentY = np.linspace(0.85, 0.15, AgentCount)
TaskY = np.linspace(0.92, 0.08, TaskCount)

TaskColors = plt.cm.Set2(np.linspace(0, 1, TaskCount))

# ---- Figure 1: Deterministic MKP structure ----
Fig1, Ax1 = plt.subplots(figsize=(9.5, 5.5))

for AgentIndex, AgentName in enumerate(AgentLabels):
    Ax1.add_patch(mpatches.FancyBboxPatch(
        (AgentX - 0.02, AgentY[AgentIndex] - 0.04), 0.16, 0.08,
        boxstyle="round,pad=0.01", facecolor="#1F4E5F", alpha=0.15,
        edgecolor="#1F4E5F", linewidth=1.5))
    Ax1.scatter(AgentX, AgentY[AgentIndex], s=160, color="#1F4E5F",
                zorder=5, edgecolors="white", linewidth=1.2)
    Ax1.text(AgentX + 0.04, AgentY[AgentIndex] + 0.015, AgentName,
             va="bottom", fontsize=9, fontweight="bold", color="#1F4E5F")
    Ax1.text(AgentX + 0.04, AgentY[AgentIndex] - 0.025,
             f"$C_{AgentIndex+1}={Capacities[AgentIndex]:.1f}$",
             va="top", fontsize=8, color="#0B1D26")

for TaskIndex, TaskName in enumerate(TaskLabels):
    Ax1.scatter(TaskX, TaskY[TaskIndex], s=110, color=TaskColors[TaskIndex],
                zorder=5, edgecolors="#0B1D26", linewidth=0.8)
    Ax1.text(TaskX - 0.04, TaskY[TaskIndex], TaskName,
             va="center", ha="right", fontsize=8.5, color="#0B1D26")

ReturnNorm = (Returns - Returns.min()) / (Returns.max() - Returns.min() + 1e-6)

for TaskIndex in range(TaskCount):
    AgentIndex = Assignments[TaskIndex]
    LineWidth = 0.8 + 2.5 * (Sizes[TaskIndex] / Sizes.max())
    EdgeColor = plt.cm.viridis(ReturnNorm[TaskIndex])
    Ax1.annotate("", xy=(AgentX - 0.025, AgentY[AgentIndex]),
                 xytext=(TaskX + 0.025, TaskY[TaskIndex]),
                 arrowprops=dict(arrowstyle="-|>", color=EdgeColor,
                                 lw=LineWidth, alpha=0.75,
                                 connectionstyle="arc3,rad=0.05"))
    MidX = 0.5 * (TaskX + AgentX)
    MidY = 0.5 * (TaskY[TaskIndex] + AgentY[AgentIndex])
    Ax1.text(MidX, MidY + 0.02,
             f"$s={Sizes[TaskIndex]:.1f}$  $r={Returns[TaskIndex]:.1f}$",
             fontsize=6.5, ha="center", va="bottom", color="#0B1D26",
             bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none",
                       alpha=0.8))

SmColorbar = plt.cm.ScalarMappable(
    cmap="viridis",
    norm=plt.Normalize(vmin=Returns.min(), vmax=Returns.max()))
SmColorbar.set_array([])
Cbar = Fig1.colorbar(SmColorbar, ax=Ax1, fraction=0.025, pad=0.02)
Cbar.set_label("Expected return $\\mathbb{E}[R_{ij}]$", fontsize=8)

Ax1.text(TaskX, 0.98, "Tasks  $\\mathcal{I}$", ha="center", fontsize=10,
         fontweight="bold", color="#E09F3E")
Ax1.text(AgentX + 0.05, 0.98, "Agents  $\\mathcal{J}$", ha="center",
         fontsize=10, fontweight="bold", color="#1F4E5F")

Ax1.set_title("Multiple Knapsack: Bipartite Assignment Structure",
               fontsize=12, fontweight="bold", pad=12)
Ax1.set_xlim(0.02, 1.05)
Ax1.set_ylim(-0.02, 1.05)
Ax1.axis("off")
Fig1.tight_layout()
Fig1.savefig(OutputDir / "mkp_structure.pdf", bbox_inches="tight")
plt.close(Fig1)

# ---- Figure 2: Stochastic sizes and returns ----
Fig2, Ax2 = plt.subplots(figsize=(9.5, 5.5))

for AgentIndex, AgentName in enumerate(AgentLabels):
    Ax2.add_patch(mpatches.FancyBboxPatch(
        (AgentX - 0.02, AgentY[AgentIndex] - 0.04), 0.16, 0.08,
        boxstyle="round,pad=0.01", facecolor="#1F4E5F", alpha=0.15,
        edgecolor="#1F4E5F", linewidth=1.5))
    Ax2.scatter(AgentX, AgentY[AgentIndex], s=160, color="#1F4E5F",
                zorder=5, edgecolors="white", linewidth=1.2)
    Ax2.text(AgentX + 0.04, AgentY[AgentIndex],
             f"{AgentName}  $C_{AgentIndex+1}={Capacities[AgentIndex]:.1f}$",
             va="center", fontsize=8.5, fontweight="bold", color="#1F4E5F")

for TaskIndex, TaskName in enumerate(TaskLabels):
    Ax2.scatter(TaskX, TaskY[TaskIndex], s=110, color=TaskColors[TaskIndex],
                zorder=5, edgecolors="#0B1D26", linewidth=0.8)
    Ax2.text(TaskX - 0.04, TaskY[TaskIndex], TaskName,
             va="center", ha="right", fontsize=8.5, color="#0B1D26")

ScenarioCount = 5
for TaskIndex in range(TaskCount):
    AgentIndex = Assignments[TaskIndex]
    MeanSize = Sizes[TaskIndex]
    MeanReturn = Returns[TaskIndex]
    SizeStd = RandomState.uniform(0.15, 0.45)
    ReturnStd = RandomState.uniform(0.3, 1.0)
    BaseColor = plt.cm.magma(ReturnNorm[TaskIndex])

    for ScenarioIndex in range(ScenarioCount):
        SampleSize = max(0.3, MeanSize + RandomState.normal(0, SizeStd))
        SampleReturn = max(0.5, MeanReturn + RandomState.normal(0, ReturnStd))
        LineWidth = 0.5 + 2.0 * (SampleSize / Sizes.max())
        AlphaVal = 0.15 + 0.12 * ScenarioIndex
        RadVal = 0.03 * (ScenarioIndex - ScenarioCount / 2)
        Ax2.annotate("",
                     xy=(AgentX - 0.025, AgentY[AgentIndex]),
                     xytext=(TaskX + 0.025, TaskY[TaskIndex]),
                     arrowprops=dict(arrowstyle="-", color=BaseColor,
                                     lw=LineWidth, alpha=AlphaVal,
                                     connectionstyle=f"arc3,rad={RadVal:.3f}"))

    MidX = 0.5 * (TaskX + AgentX)
    MidY = 0.5 * (TaskY[TaskIndex] + AgentY[AgentIndex])
    Ax2.text(MidX, MidY + 0.018,
             f"$\\bar{{r}}={MeanReturn:.1f}\\pm{ReturnStd:.1f}$\n"
             f"$\\bar{{s}}={MeanSize:.1f}\\pm{SizeStd:.1f}$",
             fontsize=6, ha="center", va="bottom", color="#0B1D26",
             bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none",
                       alpha=0.85), linespacing=1.3)

Ax2.text(TaskX, 0.98, "Tasks  $\\mathcal{I}$", ha="center", fontsize=10,
         fontweight="bold", color="#E09F3E")
Ax2.text(AgentX + 0.05, 0.98, "Agents  $\\mathcal{J}$", ha="center",
         fontsize=10, fontweight="bold", color="#1F4E5F")

Ax2.set_title("Stochastic MKP: Uncertain Sizes and Returns on Assignments",
               fontsize=12, fontweight="bold", pad=12)
Ax2.set_xlim(0.02, 1.05)
Ax2.set_ylim(-0.02, 1.05)
Ax2.axis("off")
Fig2.tight_layout()
Fig2.savefig(OutputDir / "mkp_stochastic.pdf", bbox_inches="tight")
plt.close(Fig2)

print("Figures saved: mkp_structure.pdf, mkp_stochastic.pdf")

Figures saved: mkp_structure.pdf, mkp_stochastic.pdf


# 5. Compile and Clean Beamer PDF


In [ ]:
from pathlib import Path
import subprocess

Root = Path.cwd()
if not (Root / "Presentation").exists():
    Root = Root.parent

PresentationDir = Root / "Presentation"

BuildCommand = ["latexmk", "-xelatex", "SLM.tex"]
CleanCommand = ["latexmk", "-c"]

BuildResult = subprocess.run(BuildCommand, cwd=PresentationDir, capture_output=True, text=True)
CleanResult = subprocess.run(CleanCommand, cwd=PresentationDir, capture_output=True, text=True)

BuildResult.returncode, CleanResult.returncode
